# Bushfire LGA Summary — Spatial Aggregation Pipeline

This notebook reads bushfires from `bushfire_extents`, finds which LGA each one falls in by point-in-polygon against `lga_boundaries`, and writes a per-LGA summary back to the DB.

## Output columns (`bushfire_lga_summary`)
- `lga_code`, `lga_name`, `state_name` — from `lga_boundaries`
- `bushfire_count` — number of bushfires whose centroid falls inside the LGA polygon
- `earliest_date` / `latest_date` — min / max `ignition_date` for that LGA

## How the spatial join works
PostGIS isn't available on this Postgres 18 instance, so the join is done in Python with GeoPandas — same pattern as the spatial sanity-check at the end of the `lga_boundaries` notebook:

1. Read bushfire centroids → pandas DataFrame
2. Read LGA WKT geometries → reconstitute a GeoDataFrame via `gpd.GeoSeries.from_wkt`, apply CRS `EPSG:7844` (GDA2020) — the CRS is not stored in the column
3. Build a points GeoDataFrame from the bushfire centroids in the same CRS
4. `gpd.sjoin(points, polygons, predicate='within')` — GeoPandas builds an R-tree under the hood, so this is fast even for 345k points × 547 polygons
5. Group by LGA and aggregate count + min/max ignition_date

## CRS note
Bushfires were originally captured in EPSG:4283 (GDA94); LGAs are EPSG:7844 (GDA2020). The datum shift between them is sub-metre across Australia, well inside the ~100 m simplification tolerance applied to the LGA polygons, so the join is done in 7844 directly with no reprojection.

## Time window
Only bushfires from the last `YEARS_BACK` years (relative to the run time, via `NOW()`) are included. Set `YEARS_BACK = None` to include the full history.

## Re-runs
The output table is TRUNCATEd before insert, so the pipeline is idempotent — `UNIQUE (lga_code)` won't trip on a second run.

In [12]:
from pathlib import Path
import os

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Only include bushfires from the last N years (relative to NOW()).
# Set to None to include the full history.
YEARS_BACK = 10

In [13]:
# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)
print("Connected to:", DB_NAME)

Connected to: carelink


In [14]:
# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/bushfire_lga_summary.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


# Read source data

In [15]:
# -----------------------------
# 3a. READ BUSHFIRES
# -----------------------------
# Only pull the columns we need for the join + aggregation.
# Filter by YEARS_BACK if set — make_interval(years => :n) lets us bind an int
# parameter (plain INTERVAL 'N years' requires a literal).
if YEARS_BACK is None:
    sql = """
        SELECT id, ignition_date, centroid_lat, centroid_lon
        FROM bushfire_extents
        WHERE centroid_lat IS NOT NULL
          AND centroid_lon IS NOT NULL
    """
    params = {}
else:
    sql = """
        SELECT id, ignition_date, centroid_lat, centroid_lon
        FROM bushfire_extents
        WHERE centroid_lat IS NOT NULL
          AND centroid_lon IS NOT NULL
          AND ignition_date >= NOW() - make_interval(years => :years_back)
    """
    params = {"years_back": int(YEARS_BACK)}

bushfires_df = pd.read_sql(text(sql), engine, params=params)

print(f"Bushfires read: {len(bushfires_df):,}")
if len(bushfires_df):
    print(f"Date range    : {bushfires_df['ignition_date'].min()}  →  {bushfires_df['ignition_date'].max()}")
bushfires_df.head()

Bushfires read: 45,650
Date range    : 2016-05-14 00:00:00+00:00  →  2025-09-20 00:00:00+00:00


,id,ignition_date,centroid_lat,centroid_lon
0,6bf78ede-380e-4a1f-8a04-65a7cf695bc3,2016-08-21 00:00:00+00:00,-35.312695,149.102192
1,05ef2353-deac-47e2-bda6-edd17d3d5e20,2016-11-03 00:00:00+00:00,-35.328705,149.009532
2,ff1afa34-71ea-4e6e-8eb6-d617f0e656d1,2016-11-04 00:00:00+00:00,-35.407490,149.140648
3,711c3ae9-61ee-4b2a-83c6-23681dbe6460,2016-11-21 00:00:00+00:00,-35.165783,150.678116
4,ee4790aa-afeb-4f6b-a15a-3cfed291f5b4,2017-03-09 00:00:00+00:00,-35.132376,149.140762


In [16]:
# -----------------------------
# 3b. READ LGAs AND REBUILD GEOMETRY
# -----------------------------
# geom_wkt is plain TEXT — parse it back to shapely and re-apply the CRS
# (EPSG:7844 / GDA2020), which is not stored in the column.
lgas_df = pd.read_sql(
    "SELECT lga_code, lga_name, state_name, geom_wkt FROM lga_boundaries",
    engine,
)

lgas_gdf = gpd.GeoDataFrame(
    lgas_df.drop(columns=["geom_wkt"]),
    geometry=gpd.GeoSeries.from_wkt(lgas_df["geom_wkt"]),
    crs="EPSG:7844",
)

print(f"LGAs read: {len(lgas_gdf):,}")
print(f"CRS      : {lgas_gdf.crs}")
lgas_gdf.head()

LGAs read: 547
CRS      : EPSG:7844


,lga_code,lga_name,state_name,geometry
0,10050,Albury,New South Wales,"MULTIPOLYGON (((146.86566 -36.07292, 146.86409..."
1,10180,Armidale,New South Wales,"MULTIPOLYGON (((152.38816 -30.5264, 152.38619 ..."
2,10250,Ballina,New South Wales,"MULTIPOLYGON (((153.57117 -28.87274, 153.57198..."
3,10300,Balranald,New South Wales,"MULTIPOLYGON (((142.99242 -33.85356, 143.03952..."
4,10470,Bathurst,New South Wales,"MULTIPOLYGON (((149.84877 -33.52784, 149.8363 ..."


# Spatial join + aggregation

In [17]:
# -----------------------------
# 4a. BUILD POINTS GEODATAFRAME
# -----------------------------
# points_from_xy takes (lon, lat) order — longitude FIRST. CRS must match the LGAs.
bushfires_gdf = gpd.GeoDataFrame(
    bushfires_df[["id", "ignition_date"]],
    geometry=gpd.points_from_xy(bushfires_df["centroid_lon"], bushfires_df["centroid_lat"]),
    crs="EPSG:7844",
)

print(f"Bushfire points: {len(bushfires_gdf):,}")

Bushfire points: 45,650


In [18]:
# -----------------------------
# 4b. POINT-IN-POLYGON SPATIAL JOIN
# -----------------------------
# 'within' = point falls inside the polygon. Inner join means points outside
# every LGA (offshore fires, external territories, etc.) are dropped.
# GeoPandas uses an R-tree spatial index internally — fast.
joined = gpd.sjoin(
    bushfires_gdf,
    lgas_gdf,
    how="inner",
    predicate="within",
)

matched   = len(joined)
unmatched = len(bushfires_gdf) - matched
print(f"Bushfires matched to an LGA: {matched:,}")
print(f"Bushfires outside all LGAs : {unmatched:,}")
joined[["id", "ignition_date", "lga_code", "lga_name", "state_name"]].head()

Bushfires matched to an LGA: 45,626
Bushfires outside all LGAs : 24


,id,ignition_date,lga_code,lga_name,state_name
0,6bf78ede-380e-4a1f-8a04-65a7cf695bc3,2016-08-21 00:00:00+00:00,89399,Unincorporated ACT,Australian Capital Territory
1,05ef2353-deac-47e2-bda6-edd17d3d5e20,2016-11-03 00:00:00+00:00,89399,Unincorporated ACT,Australian Capital Territory
2,ff1afa34-71ea-4e6e-8eb6-d617f0e656d1,2016-11-04 00:00:00+00:00,89399,Unincorporated ACT,Australian Capital Territory
3,711c3ae9-61ee-4b2a-83c6-23681dbe6460,2016-11-21 00:00:00+00:00,99399,Unincorp. Other Territories,Other Territories
4,ee4790aa-afeb-4f6b-a15a-3cfed291f5b4,2017-03-09 00:00:00+00:00,18710,Yass Valley,New South Wales


In [19]:
# -----------------------------
# 4c. AGGREGATE PER LGA
# -----------------------------
summary_df = (
    joined.groupby(["lga_code", "lga_name", "state_name"], dropna=False)
          .agg(
              bushfire_count=("id", "count"),
              earliest_date=("ignition_date", "min"),
              latest_date=("ignition_date", "max"),
          )
          .reset_index()
          .sort_values("bushfire_count", ascending=False)
          .reset_index(drop=True)
)

print(f"LGAs with at least one bushfire: {len(summary_df):,}")
print(f"Sum of bushfire_count           : {int(summary_df['bushfire_count'].sum()):,}  (sanity: should equal matched above)")
summary_df.head(10)

LGAs with at least one bushfire: 444
Sum of bushfire_count           : 45,626  (sanity: should equal matched above)


,lga_code,lga_name,state_name,bushfire_count,earliest_date,latest_date
0,53220,East Pilbara,Western Australia,4036,2016-05-18 00:00:00+00:00,2025-05-06 00:00:00+00:00
1,50250,Ashburton,Western Australia,2686,2016-08-25 00:00:00+00:00,2025-05-27 00:00:00+00:00
2,22110,East Gippsland,Victoria,2044,2016-05-25 00:00:00+00:00,2025-05-19 00:00:00+00:00
3,54970,Laverton,Western Australia,1379,2016-11-05 00:00:00+00:00,2023-12-29 00:00:00+00:00
4,57280,Port Hedland,Western Australia,1341,2016-08-27 00:00:00+00:00,2022-03-13 00:00:00+00:00
5,55390,Menzies,Western Australia,952,2016-08-02 00:00:00+00:00,2025-05-05 00:00:00+00:00
6,24780,Mildura,Victoria,862,2016-06-08 00:00:00+00:00,2025-06-26 00:00:00+00:00
7,11730,Clarence Valley,New South Wales,687,2016-05-15 00:00:00+00:00,2025-07-01 00:00:00+00:00
8,32500,Cook,Queensland,570,2016-05-16 00:00:00+00:00,2023-12-10 00:00:00+00:00
9,59250,Wiluna,Western Australia,557,2016-05-25 00:00:00+00:00,2024-09-11 00:00:00+00:00


# Insert into DB

In [20]:
# -----------------------------
# 5. PREPARE DATAFRAME
# -----------------------------
df_insert = summary_df.copy()

target_columns = [
    "lga_code", "lga_name", "state_name",
    "bushfire_count", "earliest_date", "latest_date",
]
df_insert = df_insert[target_columns].copy()

# text columns
for col in ["lga_code", "lga_name", "state_name"]:
    df_insert[col] = df_insert[col].astype("string").str.strip()

# integer column
df_insert["bushfire_count"] = pd.to_numeric(df_insert["bushfire_count"], errors="coerce").astype("Int64")

# datetime columns (tz-aware to match TIMESTAMPTZ)
for col in ["earliest_date", "latest_date"]:
    df_insert[col] = pd.to_datetime(df_insert[col], errors="coerce", utc=True)

print(f"Ready to insert {len(df_insert):,} rows")
df_insert.head()

Ready to insert 444 rows


,lga_code,lga_name,state_name,bushfire_count,earliest_date,latest_date
0,53220,East Pilbara,Western Australia,4036,2016-05-18 00:00:00+00:00,2025-05-06 00:00:00+00:00
1,50250,Ashburton,Western Australia,2686,2016-08-25 00:00:00+00:00,2025-05-27 00:00:00+00:00
2,22110,East Gippsland,Victoria,2044,2016-05-25 00:00:00+00:00,2025-05-19 00:00:00+00:00
3,54970,Laverton,Western Australia,1379,2016-11-05 00:00:00+00:00,2023-12-29 00:00:00+00:00
4,57280,Port Hedland,Western Australia,1341,2016-08-27 00:00:00+00:00,2022-03-13 00:00:00+00:00


In [21]:
# -----------------------------
# 6. INSERT INTO DATABASE
# -----------------------------
# TRUNCATE first so re-runs are idempotent — UNIQUE(lga_code) would otherwise fail.
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE bushfire_lga_summary"))

df_insert.to_sql(
    name="bushfire_lga_summary",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500,
)

print(f"Inserted {len(df_insert):,} rows into bushfire_lga_summary")

Inserted 444 rows into bushfire_lga_summary


In [22]:
# -----------------------------
# 7. VALIDATE
# -----------------------------
with engine.begin() as conn:
    row_count   = conn.execute(text("SELECT COUNT(*) FROM bushfire_lga_summary")).scalar()
    total_fires = conn.execute(text("SELECT SUM(bushfire_count) FROM bushfire_lga_summary")).scalar()
    top10 = pd.read_sql(
        text(
            """
            SELECT lga_name, state_name, bushfire_count, earliest_date, latest_date
            FROM bushfire_lga_summary
            ORDER BY bushfire_count DESC
            LIMIT 10
            """
        ),
        conn,
    )

print(f"Rows in bushfire_lga_summary: {row_count:,}")
print(f"Sum of bushfire_count       : {int(total_fires):,}")
print()
print("Top 10 LGAs by bushfire count:")
top10

Rows in bushfire_lga_summary: 444
Sum of bushfire_count       : 45,626

Top 10 LGAs by bushfire count:


,lga_name,state_name,bushfire_count,earliest_date,latest_date
0,East Pilbara,Western Australia,4036,2016-05-18 00:00:00+00:00,2025-05-06 00:00:00+00:00
1,Ashburton,Western Australia,2686,2016-08-25 00:00:00+00:00,2025-05-27 00:00:00+00:00
2,East Gippsland,Victoria,2044,2016-05-25 00:00:00+00:00,2025-05-19 00:00:00+00:00
3,Laverton,Western Australia,1379,2016-11-05 00:00:00+00:00,2023-12-29 00:00:00+00:00
4,Port Hedland,Western Australia,1341,2016-08-27 00:00:00+00:00,2022-03-13 00:00:00+00:00
5,Menzies,Western Australia,952,2016-08-02 00:00:00+00:00,2025-05-05 00:00:00+00:00
6,Mildura,Victoria,862,2016-06-08 00:00:00+00:00,2025-06-26 00:00:00+00:00
7,Clarence Valley,New South Wales,687,2016-05-15 00:00:00+00:00,2025-07-01 00:00:00+00:00
8,Cook,Queensland,570,2016-05-16 00:00:00+00:00,2023-12-10 00:00:00+00:00
9,Wiluna,Western Australia,557,2016-05-25 00:00:00+00:00,2024-09-11 00:00:00+00:00
